# M19 — SRQ panel-size system benchmark on Tesla T4

This notebook is synthetic-only and never downloads or opens a dataset, feature cache, validation split, or test split. It benchmarks the real P2B codec and blocked-QR implementation at widths 10,000/20,000 with 5,000 update rows. Select a **T4 GPU**, then run all cells from top to bottom. Ten atomic units are checkpointed under `/content/srq_m19/output/units`.


In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='aa8c2d016657b13bd0cb1668be8691582b2a49c3'
WORK_DIR='/content/SOHO-CL'
RUN_ROOT='/content/srq_m19'
OUTPUT_DIR=RUN_ROOT+'/output'
CONFIG='configs/srq_generalization_m19_panel_system_benchmark.json'
RUNNER='tools/srq_generalization_m19.py'
FINAL_EXPORT='/content/srq_generalization_m19_panel_system_benchmark_t4.zip'


In [ ]:
# Clean pinned checkout; source hashes are newline-normalized for Windows/Linux portability.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
def sha_raw(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
def run_visible(command):
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1','PYTHONDONTWRITEBYTECODE':'1'})
    assert process.stdout is not None
    for line in process.stdout: print(line,end='')
    returncode=process.wait()
    if returncode: raise RuntimeError(f'Command failed ({returncode}): {command}')
os.chdir('/content')
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--no-checkout','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip()==REPO_COMMIT
os.chdir(WORK_DIR)
import torch
assert torch.cuda.is_available(),'Enable a Colab GPU and restart from cell 1.'
gpu_name=torch.cuda.get_device_name(0); gpu_bytes=torch.cuda.get_device_properties(0).total_memory
assert 'T4' in gpu_name,f'M19 paper measurement requires Tesla T4; found {gpu_name!r}.'
assert gpu_bytes>=14_000_000_000,f'M19 requires at least 14 GB GPU memory; found {gpu_bytes/2**30:.2f} GiB.'
EXPECTED_SOURCE={
 'configs/srq_generalization_m19_panel_system_benchmark.json':'c3a09d5541c57b18cfb71c7adc608c2c09282e2986b9f485f2d7f2dbdfa15e2c',
 'tools/srq_generalization_m19.py':'e88e77e6a9761882621663521020827981ff9e9de5e6052e54dba64781ce0cf0',
 'tests/test_srq_generalization_m19.py':'2359bab893eca4d53b8ad87fa586b4270dbb4e157430456f325c8beaf2ec09fc',
 'docs/research/SRQ_GENERALIZATION_M19_PROTOCOL.md':'2ad09608d983ca8cedd9e7328d9e2ae756a3670553ed044c25ba57a3a5f5edcd'}
for relative,expected in EXPECTED_SOURCE.items(): assert sha_source(relative)==expected,(relative,sha_source(relative),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
print('M19 PINNED SOURCE: PASS | GPU:',gpu_name,f'{gpu_bytes/2**30:.2f} GiB')


In [ ]:
# Focused CPU correctness tests run before the long CUDA benchmark.
run_visible([sys.executable,'-B','-m','pytest','-vv','--tb=long','-p','no:cacheprovider','tests/test_srq_generalization_m19.py'])
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M19 PREFLIGHT TESTS: PASS')


## Long cell: 10 atomic units

Each subprocess completes one width/panel unit and writes its JSON atomically before the next starts. Rerunning this cell in the same runtime skips completed units. Expected wall time is hardware/load dependent; do not change repetitions or the panel grid after observing timings.


In [ ]:
# Run exactly one new unit per subprocess so completed work is resumable.
unit_dir=Path(OUTPUT_DIR)/'units'; unit_dir.mkdir(parents=True,exist_ok=True)
while len(list(unit_dir.glob('width_*_panel_*.json'))) < 10:
    before=len(list(unit_dir.glob('width_*_panel_*.json')))
    print(f'M19 START/RESUME: {before}/10 units complete',flush=True)
    run_visible([sys.executable,'-B',RUNNER,'--config',CONFIG,'--output-dir',OUTPUT_DIR,'--device','cuda','--max-new-units','1'])
    after=len(list(unit_dir.glob('width_*_panel_*.json')))
    assert after==before+1,f'Expected one new atomic unit; got {before} -> {after}'
    print(f'M19 CHECKPOINT: {after}/10 complete',flush=True)
assert Path(OUTPUT_DIR,'m19_results.json').is_file()
print('M19 ALL 10 UNITS COMPLETE')


In [ ]:
# Inspect and export every result even if a numerical gate reports a warning.
from google.colab import files
result_path=Path(OUTPUT_DIR)/'m19_results.json'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SELECTED PANEL:',result['selection']['selected_panel_size'])
print('BOTTLENECK:',json.dumps(result['bottleneck'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
manifest={'schema_version':1,'study_id':result['study_id'],'repo_commit':REPO_COMMIT,'config_sha256':EXPECTED_SOURCE[CONFIG],'runner_sha256':EXPECTED_SOURCE[RUNNER],'result_sha256':sha_raw(result_path),'status':result['status']}
with zipfile.ZipFile(FINAL_EXPORT,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in sorted(Path(OUTPUT_DIR).rglob('*')):
        if path.is_file(): archive.write(path,'results/'+str(path.relative_to(OUTPUT_DIR)).replace('\\','/'))
    for relative in (CONFIG,'docs/research/SRQ_GENERALIZATION_M19_PROTOCOL.md'):
        archive.write(relative,'source/'+relative)
    archive.writestr('M19_ARTIFACT_MANIFEST.json',json.dumps(manifest,indent=2)+'\n')
print('FINAL EXPORT:',FINAL_EXPORT,'SHA256:',sha_raw(FINAL_EXPORT),'SIZE:',Path(FINAL_EXPORT).stat().st_size)
files.download(FINAL_EXPORT)
if result['status']!='PASS_M19_PANEL_SYSTEM_BENCHMARK_T4': print('WARNING: preserve the artifact; do not relax gates or delete a panel after seeing timings.')
